In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

data = pd.read_csv("Telecom_Customer_Churn.csv")

print(data.head())
print(data.info())
print(data.shape)

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape

In [5]:
# Check missing values
print("\nMissing Values:")
print(data.isnull().sum())


# 3. Handle missing values

# Fill numerical columns with median
num_cols = data.select_dtypes(include=np.number).columns
data[num_cols] = data[num_cols].fillna(data[num_cols].median())

# Fill categorical columns with mode
cat_cols = data.select_dtypes(include="object").columns
for col in cat_cols:
    data[col] = data[col].fillna(data[col].mode()[0])



Missing Values:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


In [7]:
# 4. Remove duplicate records
data = data.drop_duplicates()

print("\nAfter removing duplicates:", data.shape)


After removing duplicates: (10, 21)


In [8]:
# 5. Standardize inconsistent data

# Remove extra spaces from text columns
for col in data.select_dtypes(include="object").columns:
    data[col] = data[col].str.strip()

# Example: standardize gender values if present
if "Gender" in data.columns:
    data["Gender"] = data["Gender"].str.lower()


In [9]:
# 6. Convert columns to correct data types

# Convert TotalCharges to numeric if present
if "TotalCharges" in data.columns:
    data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")

# Convert SeniorCitizen to integer if present
if "SeniorCitizen" in data.columns:
    data["SeniorCitizen"] = data["SeniorCitizen"].astype(int)


In [11]:
# 7. Handle outliers using IQR method

num_cols = data.select_dtypes(include=np.number).columns

for col in num_cols:
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    # Replace outliers with boundary values
    data[col] = data[col].clip(lower, upper)

In [13]:
# 8. Feature Engineering

# Create average monthly charge if TotalCharges and tenure exist
if "TotalCharges" in data.columns and "tenure" in data.columns:
    data["AvgMonthlyCharge"] = data["TotalCharges"] / (data["tenure"] + 1)

# Create tenure category
if "tenure" in data.columns:
    data["TenureGroup"] = pd.cut(
        data["tenure"],
        bins=[0, 12, 24, 48, 100],
        labels=["New", "Medium", "Long", "Very Long"]
    )


In [14]:
# 9. Normalize numerical data

num_cols = data.select_dtypes(include=np.number).columns

scaler = StandardScaler()
data[num_cols] = scaler.fit_transform(data[num_cols])


# Final dataset
print("\nFinal Dataset:")
print(data.head())

print("\nFinal Shape:", data.shape)


Final Dataset:
   customerID  gender  SeniorCitizen Partner Dependents    tenure  \
0  7590-VHVEG  Female            0.0     Yes         No -1.034800   
1  5575-GNVDE    Male            0.0      No         No  0.639141   
2  3668-QPYBK    Male            0.0      No         No -0.984075   
3  7795-CFOCW    Male            0.0      No         No  1.197122   
4  9237-HQITU  Female            0.0      No         No -0.984075   

  PhoneService     MultipleLines InternetService OnlineSecurity  ...  \
0           No  No phone service             DSL             No  ...   
1          Yes                No             DSL            Yes  ...   
2          Yes                No             DSL            Yes  ...   
3           No  No phone service             DSL            Yes  ...   
4          Yes                No     Fiber optic             No  ...   

  StreamingTV StreamingMovies        Contract PaperlessBilling  \
0          No              No  Month-to-month              Yes   
1   